In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
import gc
import json
import joblib

import pandas as pd
import numpy as np
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.utils import class_weight
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from lightgbm import LGBMClassifier

from utils import reduce_mem_usage

In [3]:
with open(r'prep_data/unique_dict.txt', 'r', encoding='utf-8') as f:
    dict_key = json.load(f)
columns_for_enc = [dict_key[col] for col in dict_key.keys()]

In [4]:
feature_names = [f'{key}_{value}' for key, values in dict_key.items() for value in values[1:]]

In [5]:
feature_names_std = [name + '_std' for name in feature_names]

In [6]:
cat_cols = list(dict_key.keys())

In [7]:
model = joblib.load(r'models/automl_model.pkl')

In [8]:
df_norm = pd.read_csv('stats.csv', index_col=0)

In [9]:
class MemoryReduceTransformer(BaseEstimator, TransformerMixin):
    
    def __init__(self):
        self.stats_ = None

    def fit(self, X, y):
        self.stats_ = reduce_mem_usage(X)
        return self

    def transform(self, X):
        X = self.stats_
        return X

In [10]:
class AggregationTransformer(BaseEstimator, TransformerMixin):
    
    def __init__(self, feature_names: list, df_norm: pd.DataFrame, group_col: str='id', agg_func: str='sum'):
        self.feature_names = feature_names
        self.group_col = group_col
        self.agg_func = agg_func
        self.df_norm = df_norm

    def fit(self, X, y):
        return self

    def transform(self, X):
        df_temp = pd.DataFrame(X, columns=self.feature_names).astype('int8')
        df_temp = df_temp.groupby(self.group_col, as_index=False).agg(self.agg_func)
        for column in df_temp.columns:
            mean = self.df_norm.loc['mean', column]
            std = self.df_norm.loc['std', column]
            df_temp = (df_temp - mean)/std
        return df_temp

In [11]:
class TrainedAutoMLWrapper(BaseEstimator, TransformerMixin):
    def __init__(self, automl_model, names):
        self.automl_model = automl_model
        self.names = names

    def fit(self, X, y=None):
        return self  

    def predict(self, X):
        df = pd.DataFrame(X, columns=self.names)
        df = df[self.automl_model.reader.used_features]
        return self.automl_model.predict(df)

In [13]:
categorical_transformer = Pipeline(steps=[
    #('red_mem', MemoryReduceTransformer()),
    ('onehot', OneHotEncoder(categories=columns_for_enc, sparse_output=False, handle_unknown='ignore', drop='first')),
    ('agg_trans', AggregationTransformer(feature_names=feature_names, df_norm=df_norm)),
    #('scaler', StandardScaler()),
])

In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, cat_cols),
    ]
)

In [15]:
clf = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', TrainedAutoMLWrapper(model, feature_names_std))
])

In [16]:
path_raw = os.path.join(os.path.abspath(os.curdir), 'raw_data')

In [17]:
files = [filename for filename in os.listdir(path_raw) if filename.startswith('train')]

In [ ]:
all_probs = {}
y = pd.read_csv(r'raw_data/target.csv')
flag_fit = False

In [ ]:
for file in files[:1]:
    file_path = os.path.join('raw_data', file)
    X = pd.read_parquet(file_path)
    print('-------------------------------------')
    print(f'Читаем файл {file_path}')
    X = reduce_mem_usage(X)
    uniqs_id = X['id'].unique()
    for idx in uniqs_id:
        print()
        print(f'Пользователь: {idx}')
        X_sample = X[X['id']==idx]
        X_sample = X_sample.drop(columns=['id'])
        print(f'Размерность X_sample: {X_sample.shape}')
        y_sample = y[y['id']==idx]
        y_sample = y_sample.drop(columns=['id'])
        if not flag_fit:
            clf.fit(X_sample, y_sample)
            flag_fit = True
        y_pred_proba = clf.predict(X_sample)
        proba = y_pred_proba.data.tolist()[0][0]
        print(proba)
        y_true = y_sample.iloc[0, 0]
        print(y_true)
        all_probs[idx] = [y_true, proba]

-------------------------------------
Читаем файл raw_data/train_data_0.pq
Используемая память до преобразования 0.897 Gb


  0%|          | 0/61 [00:00<?, ?it/s]

Используемая память после преобразования: 0.118 Gb
Количество используемой памяти уменьшено на 86.9%
